In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                                DoubleType, DateType, TimestampType)

VOL = "/Volumes/workspace/default/maplebank"

txn_schema = StructType([
    StructField("transaction_id",        StringType(),    False),
    StructField("customer_id",           StringType(),    False),
    StructField("account_id",            StringType(),    False),
    StructField("branch_id",             StringType(),    True),
    StructField("transaction_date",      DateType(),      False),
    StructField("transaction_timestamp", TimestampType(), False),
    StructField("amount_cad",            DoubleType(),    False),
    StructField("transaction_type",      StringType(),    False),
    StructField("merchant_name",         StringType(),    True),
    StructField("channel",               StringType(),    True),
])

df_txn = spark.read.option("header", True).schema(txn_schema).csv(f"{VOL}/fact_transactions.csv")
print(f"Transactions: {df_txn.count():,}")

Transactions: 10,000


In [0]:
# 1) Plain Parquet, partitioned by date
df_txn.write.mode("overwrite") \
    .partitionBy("transaction_date") \
    .parquet(f"{VOL}/formats_lab/txn_parquet")

# 2) Delta, partitioned by date
df_txn.write.format("delta").mode("overwrite") \
    .partitionBy("transaction_date") \
    .save(f"{VOL}/formats_lab/txn_delta")

print("Parquet + Delta written ✔")

Parquet + Delta written ✔


In [0]:
# See the folder-per-date structure
for f in dbutils.fs.ls(f"{VOL}/formats_lab/txn_parquet")[:8]:
    print(f.name)

_SUCCESS
transaction_date=2025-11-01/
transaction_date=2025-11-02/
transaction_date=2025-11-03/
transaction_date=2025-11-04/
transaction_date=2025-11-05/
transaction_date=2025-11-06/
transaction_date=2025-11-07/


In [0]:
def total_size(path):
    size = 0
    for f in dbutils.fs.ls(path):
        if f.isDir():
            size += total_size(f.path)
        else:
            size += f.size
    return size

csv_size     = total_size(f"{VOL}/fact_transactions.csv") if False else \
               [f.size for f in dbutils.fs.ls(VOL) if f.name == "fact_transactions.csv"][0]
parquet_size = total_size(f"{VOL}/formats_lab/txn_parquet")

print(f"CSV:     {csv_size/1024:>8.1f} KB")
print(f"Parquet: {parquet_size/1024:>8.1f} KB")
print(f"Compression ratio: {csv_size/parquet_size:.1f}x smaller")

CSV:       1018.0 KB
Parquet:    402.3 KB
Compression ratio: 2.5x smaller


In [0]:
df_pruned = spark.read.format("delta").load(f"{VOL}/formats_lab/txn_delta") \
    .filter(F.col("transaction_date") == "2025-11-15")

print(f"Rows on 2025-11-15: {df_pruned.count()}")

# The proof — look for the partition filter in the physical plan:
df_pruned.explain()

Rows on 2025-11-15: 288
== Physical Plan ==
PhotonResultStage
+- PhotonColumnarToRow
   +- PhotonProject [transaction_id#12777, customer_id#12778, account_id#12779, branch_id#12780, transaction_date#12781, transaction_timestamp#12782, amount_cad#12783, transaction_type#12784, merchant_name#12785, channel#12786]
      +- PhotonScan parquet [transaction_id#12777,customer_id#12778,account_id#12779,branch_id#12780,transaction_timestamp#12782,amount_cad#12783,transaction_type#12784,merchant_name#12785,channel#12786,transaction_date#12781] DataFilters: [], DictionaryFilters: [], Format: parquet, Location: PreparedDeltaFileIndex(1 paths)[dbfs:/Volumes/workspace/default/maplebank/formats_lab/txn_delta], OptionalDataFilters: [], PartitionFilters: [isnotnull(transaction_date#12781), (transaction_date#12781 = 2025-11-15)], ReadSchema: struct<transaction_id:string,customer_id:string,account_id:string,branch_id:string,transaction_ti..., RequiredDataFilters: []


== Photon Explanation ==
The query i

In [0]:
# Try appending a DataFrame with a surprise extra column
df_bad = df_txn.limit(10).withColumn("surprise_column", F.lit("oops"))

try:
    df_bad.write.format("delta").mode("append").save(f"{VOL}/formats_lab/txn_delta")
    print("❌ This should not have succeeded!")
except Exception as e:
    print("✅ Delta REJECTED the write — schema enforcement works")
    print(f"   Error type: {type(e).__name__}")

✅ Delta REJECTED the write — schema enforcement works
   Error type: AnalysisException


In [0]:
# Same write, but explicitly opting in to the new column
df_bad.write.format("delta").mode("append") \
    .option("mergeSchema", "true") \
    .save(f"{VOL}/formats_lab/txn_delta")

df_check = spark.read.format("delta").load(f"{VOL}/formats_lab/txn_delta")
print(f"Columns now: {len(df_check.columns)} (was 10)")
print(f"Rows now:    {df_check.count():,} (was 10,000)")
df_check.filter(F.col("surprise_column").isNotNull()).select("transaction_id", "surprise_column").show(3)

Columns now: 11 (was 10)
Rows now:    10,010 (was 10,000)
+--------------+---------------+
|transaction_id|surprise_column|
+--------------+---------------+
|    TXN1000003|           oops|
|    TXN1000010|           oops|
|    TXN1000001|           oops|
+--------------+---------------+
only showing top 3 rows


In [0]:
df_txn.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("transaction_date") \
    .save(f"{VOL}/formats_lab/txn_delta")

print(f"Reset: {spark.read.format('delta').load(f'{VOL}/formats_lab/txn_delta').count():,} rows, "
      f"{len(spark.read.format('delta').load(f'{VOL}/formats_lab/txn_delta').columns)} columns")

Reset: 10,000 rows, 10 columns
